# VoiceAI — generate the 4 hero greeting clips (Chatterbox TTS)

Run this in **Google Colab** (Runtime → Change runtime type → **GPU**).

The 3 GB model downloads onto Google's machine, generates 4 short English
greetings in seconds, and you download only the tiny MP3s (< 2 MB total).
Nothing large ever touches your laptop.

**What you get:** `greet-banking.mp3`, `greet-telecoms.mp3`,
`greet-insurance.mp3`, `greet-government.mp3` — drop them into
`web/public/assets/` in the repo and the bubbles will play them.

Chatterbox is MIT-licensed (commercial use OK). It supports zero-shot voice
cloning: upload a ~10 s reference clip and every greeting is spoken in that
voice. Skip the upload to use the model's default voice.

## 1. Confirm GPU + install Chatterbox

In [ ]:
!nvidia-smi -L  # should list a GPU (T4 is fine). If it errors, set Runtime -> GPU.
!pip -q install chatterbox-tts

## 2. Load the model

In [ ]:
import torch, torchaudio as ta
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
model = ChatterboxTTS.from_pretrained(device=device)
print("Model loaded. Sample rate:", model.sr)

## 3. (Optional) Upload a reference voice to clone

Upload a clean ~10 second WAV/MP3 of the voice you want (e.g. a warm
Botswana-accented English speaker — could be one of you two, recorded on a
phone). Every greeting will then be spoken in that voice.

**Skip this cell** to use Chatterbox's default voice.

In [ ]:
from google.colab import files
import os

REFERENCE = None
print("Pick a reference clip, or press Cancel to use the default voice.")
try:
    up = files.upload()
    if up:
        REFERENCE = list(up.keys())[0]
        print("Cloning voice from:", REFERENCE)
except Exception as e:
    print("No reference uploaded — using default voice.", e)

## 4. Generate the four greetings

Edit the wording here if you want — keep it short so it fits the hero.

In [ ]:
GREETINGS = {
    "banking":    "Hi, welcome to VoiceAI for banking. How may I help you today?",
    "telecoms":   "Hi, welcome to VoiceAI telecoms. How can I help you today?",
    "insurance":  "Hi, welcome to VoiceAI Insurance. How may I help you today?",
    "government": "Hello, welcome to VoiceAI for government services. How may I help you today?",
}

gen_kwargs = {}
if REFERENCE:
    gen_kwargs["audio_prompt_path"] = REFERENCE

for sector, text in GREETINGS.items():
    wav = model.generate(text, **gen_kwargs)
    ta.save(f"{sector}.wav", wav, model.sr)
    print("generated", sector)

## 5. Compress to small MP3s and download

MP3 keeps each clip well under a few hundred KB — tiny to download and to
commit into the repo.

In [ ]:
from google.colab import files

for sector in GREETINGS:
    # 96 kbps mono is plenty for a short greeting
    !ffmpeg -y -loglevel error -i "{sector}.wav" -ac 1 -b:a 96k "greet-{sector}.mp3"

!ls -lh greet-*.mp3
for sector in GREETINGS:
    files.download(f"greet-{sector}.mp3")

## 6. Put them in the site

Move the four `greet-*.mp3` files into **`web/public/assets/`** in the repo.
Tell Claude “the clips are in” and it will wire each bubble to play its own
clip (with the browser voice kept as a fallback).